<a href="https://colab.research.google.com/github/Gayathri288/GenAI_LAB_231801039/blob/main/GenAI_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q faiss-cpu transformers torch pandas

In [5]:
from google.colab import files
uploaded = files.upload()

Saving products_dataset.csv to products_dataset (1).csv


In [6]:
import pandas as pd

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("Dataset loaded successfully")
print(df.head())

Dataset loaded successfully
               product_name
0             iPhone 13 Pro
1        Samsung Galaxy S21
2      Dell Inspiron Laptop
3        HP Pavilion Laptop
4  Sony Wireless Headphones


In [7]:
TEXT_COLUMN = df.columns[0]   # default first column
item_texts = df[TEXT_COLUMN].astype(str).tolist()

print("\nTotal items:", len(item_texts))
print("Sample item:", item_texts[0])


Total items: 10
Sample item: iPhone 13 Pro


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import faiss

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
encoder = AutoModel.from_pretrained(model_name).to(device)
encoder.eval()


In [8]:
def encode_texts(texts, batch_size=16):
    all_embeddings = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]

            tokens = tokenizer(
                batch,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(device)

            outputs = encoder(**tokens)

            # Mean pooling
            embeddings = outputs.last_hidden_state.mean(dim=1)
            embeddings = embeddings.cpu().numpy().astype("float32")

            all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)

item_embeddings = encode_texts(item_texts)


In [9]:
faiss.normalize_L2(item_embeddings)

print("\nEmbeddings shape:", item_embeddings.shape)


Embeddings shape: (10, 384)


In [10]:
embedding_dim = item_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(item_embeddings)

print("FAISS index built with", index.ntotal, "items")


FAISS index built with 10 items


In [11]:
def get_similar_items(item_id, k=5):
    query_vector = item_embeddings[item_id:item_id+1]
    scores, indices = index.search(query_vector, k)
    return scores[0], indices[0]

In [12]:
query_id = 0

scores, idxs = get_similar_items(query_id, k=5)

print("\nQUERY ITEM:")
print(item_texts[query_id])

print("\nSIMILAR ITEMS:")
for score, idx in zip(scores, idxs):
    print(f"-> {item_texts[idx]} (score={score:.3f})")


QUERY ITEM:
iPhone 13 Pro

SIMILAR ITEMS:
-> iPhone 13 Pro (score=1.000)
-> Samsung Galaxy S21 (score=0.535)
-> OnePlus Mobile (score=0.425)
-> HP Pavilion Laptop (score=0.395)
-> Apple MacBook Air (score=0.367)
